# Documentação do pipeline em produção
Uso no Fabric, tabelas origem/destino e lógica incremental (carregar existente → filtrar novas → classificar novas → append na gold_avaliacoes_servicos_sentimento).

In [2]:
# Dependências: Groq (API), tqdm (progresso), pyodbc/sqlalchemy (Fabric ODBC), openpyxl (Excel)
%pip install groq tqdm pyodbc sqlalchemy --quiet

StatementMeta(, de9eb51b-5cd4-49f9-85c2-a2924ca2474c, 9, Finished, Available, Finished, False)


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



Config + leitura Gold

In [3]:
# =============================================================================
# CÉLULA 2: IMPORTS E CONFIGURAÇÃO
# =============================================================================
# Tabelas: origem = avaliação serviço; destino = só seqFluxo + colunas da análise (para relacionamento no Power BI).

import json
import os
import re
import unicodedata
from time import sleep
from collections import Counter
import pandas as pd
import numpy as np
from tqdm import tqdm
from groq import Groq

# ---- Tabelas no Lakehouse (Fabric) ----
# No Fabric: vincule o Lakehouse ao notebook; o nome deve bater com LAKEHOUSE_CATALOG.
GOLD_ORIGEM = "gold_avaliacoes_servico"
GOLD_SENTIMENTO = "gold_avaliacoes_servicos_sentimento"  # produção: só seqFluxo + colunas de análise
LAKEHOUSE_CATALOG = "lh_cidade_inteligente_santos"

# ---- Modo de leitura/escrita (Fabric: sempre Lakehouse) ----
# modo = False → Spark (notebook no Fabric). modo = True → ODBC (ex.: Cursor local contra DW).
# [MANUTENÇÃO] Modo arquivo local removido: no Fabric a leitura é sempre do Lakehouse.
# USAR_ARQUIVO_LOCAL = False
# ARQUIVO_ORIGEM = "gold_avaliacoes_servico.csv"
# ARQUIVO_SENTIMENTO = "gold_avaliacoes_servicos_sentimento.csv"

# ---- Colunas da origem (avaliação serviço) ----
COLUNA_CHAVE = "seqFluxo"   # número da OS; chave para relacionamento com gold_avaliacoes_serviço
COLUNA_NOTA = "classificacao_servico_prestado"
COLUNA_COMENTARIO = "obs_classificacao_de_servico_prestado"

REINCORPORAR_SEM_TEXTO_NA_SAIDA = True  # Inclui OS sem comentário na tabela de sentimento
MIN_CARACTERES_COMENTARIO = 5  # Comentário < 5 chars → classifica só pela nota (regra_nota)
SLEEP_ENTRE_CHAMADAS = 2
SLEEP_APOS_LIMITE = 65
MAX_TENTATIVAS = 5
MODELO_GROQ = "llama-3.1-8b-instant"
API_KEY = os.environ.get("GROQ_API_KEY", "")

# Ponto de extremidade do Data Warehouse do Fabric (Lakehouse lh_cidade_inteligente_santos)
CONN_STR = (
    "Driver={ODBC Driver 18 for SQL Server};"
    "Server=ena6obg6j2cevcppw7dn7yu57a-knnp5frchjbujdik4l3nmgrgsa.datawarehouse.fabric.microsoft.com;"
    "Database=lh_cidade_inteligente_santos;"
    "Authentication=ActiveDirectoryInteractive;"
    "Encrypt=yes;TrustServerCertificate=yes;"
)

_MARKER_JSON = chr(96) + chr(96) + chr(96) + "json"
_MARKER_END = chr(96) + chr(96) + chr(96)

PALAVRAS_ELOGIO = [
    "otimo", "excelente", "parabens", "perfeito", "top", "maravilhoso",
    "obrigado", "obrigada", "agradeco", "rapidez", "eficiente", "qualidade"
]
PALAVRAS_RECLAMACAO = ["pessimo", "horrivel", "vergonha", "lixo", "absurdo"]
PADROES_RECLAMACAO = [
    "nao foi feito", "nao executado", "nao fizeram", "nao resolveram",
    "nao resolveu", "nao resolvido"
]
STOPWORDS = {
    "para", "com", "que", "uma", "por", "mais", "mas", "nao", "ele", "ela",
    "estao", "esta", "foi", "sao", "ser", "tem", "sua", "como", "ate", "sobre",
    "todos", "entre", "muito", "este", "esse", "quando", "fazer", "outro", "nos",
    "hoje", "apos", "sem", "pelo", "pela"
}

print("Configuração carregada. Leitura/escrita: Lakehouse (Spark no Fabric ou ODBC).")


StatementMeta(, de9eb51b-5cd4-49f9-85c2-a2924ca2474c, 11, Finished, Available, Finished, False)

Configuração carregada. Leitura/escrita: Lakehouse (Spark no Fabric ou ODBC).


In [4]:
# =============================================================================
# CÉLULA 3: NORMALIZAÇÃO E TRATAMENTO DE RESPOSTA GROQ
# =============================================================================
# Normalização de texto (acentos), detecção de elogio/reclamação por palavra e extração de JSON da resposta Groq

def normalizar_texto(texto):
    if not texto or not isinstance(texto, str):
        return ""
    t = texto.lower().strip()
    t = unicodedata.normalize("NFD", t)
    return "".join(c for c in t if unicodedata.category(c) != "Mn")

def tem_palavra_elogio(texto_norm):
    return any(p in texto_norm for p in PALAVRAS_ELOGIO)

def tem_palavra_reclamacao(texto_norm):
    if any(p in texto_norm for p in PALAVRAS_RECLAMACAO):
        return True
    return any(p in texto_norm for p in PADROES_RECLAMACAO)

def limpar_json(texto):
    if not texto or not isinstance(texto, str):
        return "{}"
    texto = texto.strip()
    if _MARKER_JSON in texto:
        texto = texto.split(_MARKER_JSON)[1].split(_MARKER_END)[0]
    elif _MARKER_END in texto:
        texto = texto.split(_MARKER_END)[1].split(_MARKER_END)[0]
    texto = texto.strip()
    if texto.count('"') % 2 != 0:
        texto = texto + '"'
    return texto

def extrair_json_resposta(texto):
    if not texto or not isinstance(texto, str):
        return None
    t = limpar_json(texto)
    start, end = t.find("{"), t.rfind("}")
    if start != -1 and end != -1 and end > start:
        t = t[start:end + 1]
    try:
        return json.loads(t)
    except json.JSONDecodeError:
        pass
    t = re.sub(r":\s*true\b", ": true", t, flags=re.I)
    t = re.sub(r":\s*false\b", ": false", t, flags=re.I)
    try:
        return json.loads(t)
    except json.JSONDecodeError:
        return None

def obter_texto_resposta_groq(response):
    try:
        if response.choices and len(response.choices) > 0:
            content = response.choices[0].message.content
            if content:
                return content.strip()
    except (IndexError, AttributeError, TypeError):
        pass
    return ""

print("Funções de normalização e JSON carregadas.")

StatementMeta(, de9eb51b-5cd4-49f9-85c2-a2924ca2474c, 12, Finished, Available, Finished, False)

Funções de normalização e JSON carregadas.


In [5]:
# =============================================================================
# CÉLULA 4: CLASSIFICAÇÃO POR REGRAS E API GROQ
# =============================================================================
# Classificação: (1) regras (nota + palavras) (2) API Groq se incerto; se client=None usa só regras; fallbacks para JSON/rede/429

def obter_classificacao_regras(texto, nota):
    if pd.isna(nota):
        nota = 3
    else:
        nota = float(nota)
    if pd.isna(texto) or len(str(texto).strip()) < MIN_CARACTERES_COMENTARIO:
        sentimento = "positivo" if nota >= 4 else "negativo" if nota <= 2 else "neutro"
        return {
            "sentimento": sentimento, "categoria": "sem_comentario", "tema": "nao_aplicavel",
            "requer_atencao": False, "metodo": "regra_nota",
        }, True
    texto_norm = normalizar_texto(str(texto))
    if nota >= 4 and tem_palavra_elogio(texto_norm):
        return {"sentimento": "positivo", "categoria": "elogio_geral", "tema": "satisfacao_geral",
                "requer_atencao": False, "metodo": "regra_palavra"}, True
    if nota <= 2 and tem_palavra_reclamacao(texto_norm):
        return {"sentimento": "negativo", "categoria": "reclamacao_grave", "tema": "insatisfacao_grave",
                "requer_atencao": True, "metodo": "regra_palavra"}, True
    sentimento_regra = "positivo" if nota >= 4 else "negativo" if nota <= 2 else "neutro"
    return {
        "sentimento": sentimento_regra, "categoria": "outro", "tema": "outro",
        "requer_atencao": nota <= 2, "metodo": "regra_incerta",
    }, False

def mapear_categoria_api(val):
    v = str(val).lower().strip()
    if "elogio" in v:
        return "elogio_geral"
    if "reclamac" in v or "reclamação" in v.replace("ç", "c"):
        return "reclamacao_grave"
    return v if v in ("sugestao", "duvida", "outro") else "outro"

def classificar_avaliacao(client, texto, nota):
    classificacao_regras, regra_certa = obter_classificacao_regras(texto, nota)
    if regra_certa:
        classificacao_regras["sentimento_regras"] = classificacao_regras["sentimento"]
        classificacao_regras["categoria_regras"] = classificacao_regras["categoria"]
        return classificacao_regras
    # Se nao tem cliente API (ex.: API_KEY vazia), usa so o resultado das regras
    if client is None:
        classificacao_regras["sentimento_regras"] = classificacao_regras["sentimento"]
        classificacao_regras["categoria_regras"] = classificacao_regras["categoria"]
        classificacao_regras["metodo"] = "regra_incerta_sem_api"
        return classificacao_regras
    nota = 3 if pd.isna(nota) else float(nota)
    texto_limpo = str(texto).replace('"', "'").replace("\n", " ")[:500]
    sentimento_regra = classificacao_regras["sentimento"]
    categoria_regra = classificacao_regras["categoria"]
    prompt = f"""
Analise o seguinte feedback de serviço público e classifique-o.
Nota do Usuário: {nota}/5
Comentário: "{texto_limpo}"
Pré-classificação por regras: sentimento={sentimento_regra}, categoria={categoria_regra}.
Confirme ou ajuste. Responda APENAS um JSON válido (sem markdown) com: "sentimento", "categoria", "tema", "requer_atencao".
"""
    tentativa = 0
    while True:
        try:
            response = client.chat.completions.create(
                model=MODELO_GROQ,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1,
                max_tokens=256,
            )
            res_texto = obter_texto_resposta_groq(response)
            if not res_texto:
                raise ValueError("Resposta vazia")
            resultado = extrair_json_resposta(res_texto)
            if resultado is None:
                raise json.JSONDecodeError("Parse falhou", res_texto[:50], 0)
            resultado.setdefault("sentimento", sentimento_regra)
            resultado["categoria"] = mapear_categoria_api(resultado.get("categoria", "outro"))
            resultado.setdefault("tema", "outro")
            resultado.setdefault("requer_atencao", nota <= 2)
            resultado["metodo"] = "groq_api"
            resultado["sentimento_regras"] = sentimento_regra
            resultado["categoria_regras"] = categoria_regra
            sleep(SLEEP_ENTRE_CHAMADAS)
            return resultado
        except json.JSONDecodeError:
            tentativa += 1
            if tentativa >= MAX_TENTATIVAS:
                classificacao_regras["categoria"] = "erro_json"
                classificacao_regras["tema"] = "erro"
                classificacao_regras["metodo"] = "fallback_json_error"
                classificacao_regras["sentimento_regras"] = sentimento_regra
                classificacao_regras["categoria_regras"] = categoria_regra
                return classificacao_regras
            sleep(1)
        except Exception as e:
            if "429" in str(e) or "rate_limit" in str(e).lower() or "ResourceExhausted" in str(e):
                sleep(SLEEP_APOS_LIMITE)
                continue
            tentativa += 1
            if tentativa >= MAX_TENTATIVAS:
                classificacao_regras["categoria"] = "erro_api"
                classificacao_regras["tema"] = "erro"
                classificacao_regras["metodo"] = "fallback_api_error"
                classificacao_regras["sentimento_regras"] = sentimento_regra
                classificacao_regras["categoria_regras"] = categoria_regra
                return classificacao_regras
            sleep(2)
    classificacao_regras["categoria"] = "erro_desconhecido"
    classificacao_regras["tema"] = "erro"
    classificacao_regras["metodo"] = "fallback_final"
    classificacao_regras["sentimento_regras"] = sentimento_regra
    classificacao_regras["categoria_regras"] = categoria_regra
    return classificacao_regras

print("Funções de classificação carregadas.")

StatementMeta(, de9eb51b-5cd4-49f9-85c2-a2924ca2474c, 13, Finished, Available, Finished, False)

Funções de classificação carregadas.


In [6]:
# =============================================================================
# CÉLULA 5: PALAVRA FOCO
# =============================================================================
# palavra_foco: por comentário, a palavra (tokenizada, sem stopwords) de maior frequência no corpus

def tokenizar(texto, min_chars=4):
    if pd.isna(texto) or not str(texto).strip():
        return []
    t = str(texto).lower()
    t = unicodedata.normalize("NFD", t)
    t = "".join(c for c in t if unicodedata.category(c) != "Mn")
    palavras = re.findall(rf"\b[a-z]{{{min_chars},}}\b", t)
    return [p for p in palavras if p not in STOPWORDS]

def criar_coluna_palavra_foco(df, col_comentario=COLUNA_COMENTARIO):
    serie = df[col_comentario] if col_comentario in df.columns else pd.Series(dtype=object)
    todos = []
    for v in serie.dropna():
        todos.extend(tokenizar(v))
    freq = Counter(todos)
    def palavra_foco(texto):
        tokens = tokenizar(texto)
        if not tokens:
            return None
        return max(tokens, key=lambda w: freq.get(w, 0))
    return serie.apply(palavra_foco)

print("Função palavra_foco carregada.")

StatementMeta(, de9eb51b-5cd4-49f9-85c2-a2924ca2474c, 14, Finished, Available, Finished, False)

Função palavra_foco carregada.


In [7]:
# =============================================================================
# CÉLULA 6: LEITURA E ESCRITA (Lakehouse / Data Warehouse)
# =============================================================================
# No Fabric: sempre Spark (tabelas gold_avaliacoes_servico e gold_avaliacoes_servicos_sentimento).
# Fora do Fabric (ex.: Cursor): ODBC no endpoint do Data Warehouse (mesmo database).
# [MANUTENÇÃO] Modo arquivo local (CSV/Excel) removido: no Fabric a leitura é sempre do Lakehouse.

def carregar_origem():
    """Retorna (df_origem, modo). modo = False (Spark/Fabric) | True (ODBC)."""
    # [MANUTENÇÃO] Modo arquivo removido: if USAR_ARQUIVO_LOCAL and os.path.isfile(ARQUIVO_ORIGEM): ...
    # Fabric: Spark
    try:
        _ = spark
        df = spark.table(f"{LAKEHOUSE_CATALOG}.{GOLD_ORIGEM}").toPandas()
        return df, False
    except NameError:
        pass
    # Local com ODBC (ex.: conexão ao Data Warehouse)
    try:
        import pyodbc
        conn = pyodbc.connect(CONN_STR)
        df = pd.read_sql(f"SELECT * FROM dbo.{GOLD_ORIGEM}", conn)
        conn.close()
        return df, True
    except Exception:
        raise FileNotFoundError("Origem não encontrada. No Fabric: vincule o Lakehouse. Fora do Fabric: use ODBC (CONN_STR).")

def _manter_apenas_colunas_sentimento(df):
    """Mantém só seqFluxo + colunas de análise (evita colunas antigas da tabela no Lakehouse)."""
    if df is None or len(df) == 0:
        return df
    cols = [c for c in [COLUNA_CHAVE] + [x for x in df.columns if str(x).startswith("analise_") or str(x) == "palavra_foco"] if c in df.columns]
    if cols:
        return df[cols].copy()
    return df

def carregar_existente(modo):
    """Carrega a tabela de sentimento já processada (apenas colunas de interesse). modo = False (Spark) | True (ODBC)."""
    # [MANUTENÇÃO] Modo arquivo removido: if modo == "arquivo": ler CSV/Excel ...
    try:
        _ = spark
        df = spark.table(f"{LAKEHOUSE_CATALOG}.{GOLD_SENTIMENTO}").toPandas()
        return _manter_apenas_colunas_sentimento(df)
    except NameError:
        pass
    except Exception as e:
        if "TABLE_OR_VIEW_NOT_FOUND" in str(e) or "AnalysisException" in str(type(e).__name__):
            return pd.DataFrame()
        pass
    try:
        import pyodbc
        conn = pyodbc.connect(CONN_STR)
        df = pd.read_sql(f"SELECT * FROM dbo.{GOLD_SENTIMENTO}", conn)
        conn.close()
        return _manter_apenas_colunas_sentimento(df)
    except Exception:
        return pd.DataFrame()

def gravar_tabela(df_final, modo, append=True):
    """Grava a tabela de sentimento (seqFluxo + colunas da análise).
    append=True: adiciona apenas os registros passados (padrão no Fabric).
    append=False: sobrescreve a tabela. modo = False (Spark) | True (ODBC)."""
    # [MANUTENÇÃO] Modo arquivo removido: if modo == "arquivo": concat + to_csv/to_excel ...
    if modo is True:
        from sqlalchemy import create_engine
        from urllib.parse import quote_plus
        engine = create_engine(f"mssql+pyodbc:///?odbc_connect={quote_plus(CONN_STR)}")
        df_final.to_sql(GOLD_SENTIMENTO, engine, schema="dbo", if_exists="append" if append else "replace", index=False)
        engine.dispose()
    else:
        df_spark = spark.createDataFrame(df_final)
        # Sem prefixo do Lakehouse: grava na tabela do Lakehouse vinculado ao notebook
        path_tabela = f"Tables/{GOLD_SENTIMENTO}"
        if append:
            df_spark.write.format("delta").mode("append").save(path_tabela)
        else:
            df_spark.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(path_tabela)
    print(f"Tabela gravada: {GOLD_SENTIMENTO} | Registros escritos: {len(df_final)}")

print("Funções de I/O carregadas.")

StatementMeta(, de9eb51b-5cd4-49f9-85c2-a2924ca2474c, 15, Finished, Available, Finished, False)

Funções de I/O carregadas.


In [8]:
# =============================================================================
# CÉLULA 7: EXECUÇÃO - PROCESSAMENTO INCREMENTAL
# =============================================================================
# 1. Carrega origem e já processados  2. Filtra OS novas  3. Classifica  4. Cria palavra_foco  5. Grava

print("--- Processamento incremental de sentimento (v2 - regras + palavra_foco) ---")
client = Groq(api_key=API_KEY)

# Carrega tabela de origem e tabela de destino (para saber quais OS já foram processadas)
df_origem, modo = carregar_origem()
df_existente = carregar_existente(modo)

if COLUNA_NOTA not in df_origem.columns or COLUNA_COMENTARIO not in df_origem.columns:
    raise ValueError(f"Colunas obrigatórias não encontradas: {COLUNA_NOTA}, {COLUNA_COMENTARIO}")

df_origem[COLUNA_NOTA] = pd.to_numeric(df_origem[COLUNA_NOTA], errors="coerce")

# Identifica OS novas (seqFluxo que ainda não está na gold_avaliacoes_sentimento)
seq_existentes = set(df_existente[COLUNA_CHAVE].dropna().astype(int)) if len(df_existente) > 0 else set()
seq_origem = pd.to_numeric(df_origem[COLUNA_CHAVE], errors="coerce").fillna(-999999).astype(int)
df_novos = df_origem[~seq_origem.isin(seq_existentes)].copy()

if len(df_novos) == 0:
    print("Nenhuma OS nova. Tabela já está atualizada.")
else:
    print(f"OS novas: {len(df_novos)} | Já processadas: {len(seq_existentes)}")

    # Separa OS com comentário (para classificar) das sem comentário
    mask_texto = df_novos[COLUNA_COMENTARIO].notna() & (df_novos[COLUNA_COMENTARIO].astype(str).str.strip() != "")
    df_classificar = df_novos[mask_texto].copy()
    df_sem_texto = df_novos[~mask_texto].copy()

    # Classifica cada OS com comentário (cache evita chamar API para comentários repetidos)
    resultados = []
    cache = {}
    for _, row in tqdm(df_classificar.iterrows(), total=len(df_classificar), desc="Classificando"):
        texto, nota = row[COLUNA_COMENTARIO], row[COLUNA_NOTA]
        chave = (str(texto).strip().lower()[:500], None if pd.isna(nota) else float(nota))
        if chave in cache:
            resultados.append(cache[chave].copy())
        else:
            r = classificar_avaliacao(client, texto, nota)
            cache[chave] = r.copy()
            resultados.append(r)

    # Junta resultados da classificação às colunas da OS
    df_analise = pd.DataFrame(resultados)
    for col in df_analise.columns:
        df_classificar[f"analise_{col}"] = df_analise[col].values
    df_classificar["analise_status"] = "classificado"
    df_sem_texto["analise_status"] = "nao_classificado_sem_texto"
    df_novos_saida = pd.concat([df_classificar, df_sem_texto], ignore_index=True, sort=False)

    # Monta df_final: existentes + novos (com ou sem texto, conforme config)
    if REINCORPORAR_SEM_TEXTO_NA_SAIDA:
        df_final = pd.concat([df_existente, df_novos_saida], ignore_index=True, sort=False)
    else:
        df_final = pd.concat([df_existente, df_classificar], ignore_index=True, sort=False)

    df_final = df_final.drop_duplicates(subset=[COLUNA_CHAVE], keep="last")

    # Garante tipo booleano para requer_atencao
    if "analise_requer_atencao" in df_final.columns:
        df_final["analise_requer_atencao"] = df_final["analise_requer_atencao"].apply(
            lambda x: bool(x) if isinstance(x, bool) else str(x).lower() == "true"
        )

    # Cria coluna palavra_foco (palavra mais frequente no corpus por comentário) antes de gravar
    print("Criando coluna palavra_foco...")
    df_final["palavra_foco"] = criar_coluna_palavra_foco(df_final)

    # ---------------------------------------------
    # NOVO: tabela mínima de sentimento para o Lakehouse
    # ---------------------------------------------

    # Sempre manter a chave principal (seqFluxo)
    colunas_chave = [COLUNA_CHAVE]

    # Se quiser incluir outros identificadores (por exemplo número da OS),
    # adicione aqui o nome exato das colunas que existem em df_final.
    colunas_ids_extra = [
        # "numero_os",  # exemplo – ajuste para o nome real, se existir
        # "sqfluxo",    # exemplo – ajuste para o nome real, se existir
    ]

    # Todas as colunas geradas pela análise (regras/API)
    colunas_analise = [c for c in df_final.columns if c.startswith("analise_")]

    # Colunas auxiliares que fazem sentido na tabela de sentimento
    colunas_auxiliares = ["palavra_foco", "analise_status"]

    # Monta lista final de colunas, garantindo que existem em df_final
    colunas_sentimento = colunas_chave + colunas_ids_extra + colunas_analise + colunas_auxiliares
    colunas_sentimento = [c for c in colunas_sentimento if c in df_final.columns]

    df_sentimento = df_final[colunas_sentimento].copy()

    # Grava APENAS a tabela de sentimento mínima no Fabric
    gravar_tabela(df_sentimento, modo)

    # Exibe resumo da execução

    n_class = len(df_classificar)
    print("\n📊 RESUMO")
    print(f"  Novas classificadas: {n_class}")
    if n_class > 0 and "analise_metodo" in df_classificar.columns:
        print(df_classificar["analise_metodo"].value_counts().to_string())
    if n_class > 0 and "analise_sentimento" in df_classificar.columns:
        print("  Sentimento:", df_classificar["analise_sentimento"].value_counts().to_dict())
    if "palavra_foco" in df_final.columns:
        top_foco = df_final["palavra_foco"].value_counts().head(10)
        print("  Top 10 palavra_foco:", top_foco.to_dict())

StatementMeta(, de9eb51b-5cd4-49f9-85c2-a2924ca2474c, 16, Finished, Available, Finished, False)

--- Processamento incremental de sentimento (v2 - regras + palavra_foco) ---
Nenhuma OS nova. Tabela já está atualizada.


In [11]:
# Ajuste o nome se no Fabric for com catálogo (ex.: lh_cidade_inteligente_santos.gold_avaliacoes_servicos_sentimento)
df = spark.table("gold_avaliacoes_servicos_sentimento")  # ou spark.table("lh_cidade_inteligente_santos.gold_avaliacoes_servicos_sentimento")
df.groupBy("analise_status").count().show()

StatementMeta(, de9eb51b-5cd4-49f9-85c2-a2924ca2474c, 19, Finished, Available, Finished, False)

+--------------------+-----+
|      analise_status|count|
+--------------------+-----+
|        classificado| 1340|
|nao_classificado_...|12745|
+--------------------+-----+



In [14]:
%%sql
SELECT * FROM gold_avaliacoes_servicos_sentimento
WHERE analise_sentimento = 'positivo'

StatementMeta(, de9eb51b-5cd4-49f9-85c2-a2924ca2474c, 22, Finished, Available, Finished, False)

<Spark SQL result set with 365 rows and 10 fields>